# Dual-stage receive coil amplifier

In [153]:
import SLiCAP as sl
from sympy import pi, sqrt, log, Abs, N
sl.initProject('Hearing loop', notebook=True)
specs     = sl.csv2specs("fb_concept.csv")
spec_dict = sl.specs2dict(specs)
# import operating point information from the last run of 'firstStageDesign.ipynb'.
%store -r op_info 
%store -r channel
channel = channel.upper()

Compiling library: SLiCAP.lib.
Compiling library: SLiCAPmodels.lib.


## Stages 

In this notebook, we will present a dual-stage solution for the hearing-loop receive-coil amplifier. We will use a P-channel differential-pair input stage and an N-channel CS output stage.

**Questions**

1. Give at least two other possible dual-stage alternatives for the controller architecture.
2. Discuss the pros and cons of these two architectures with respect to the one discussed in this notebook.

### Input stage
P-channel differential pair with geometry and operating conditions according to the noise design

In [154]:
if channel == "P":
    g_m1     = op_info["g_m"]
    g_o1     = op_info["g_o"]
    c_iss1   = op_info["c_iss"]
else:
    print("Please run 'firstStageDesign.ipynb' with channel = 'P'")

### Ouput stage
N-channel, smallest length, width large to ensure a high gm/ID ratio (> 15) and a low saturation voltage.

Operating current $I_{D2}$ equals $Bi2$ times the required peak current:

\begin{equation}
I_{D2}=\frac{Vi_{pp}}{R_f}
\end{equation}

In [155]:
I_D2 = spec_dict["Vi_pp"].value/(2*spec_dict["R_f"].value)
sl.eqn2html("I_D2", I_D2)

### Operating point information output stage

In [156]:
W2     = 10e-6
L2     = 180e-9
M2     = 1
ID2    = I_D2
VDS2   = 0.45

params = [("ID", ID2), ("VDS", VDS2), ("W", W2), ("L", L2), ("M", M2)]
simCmd = "OP"
names  = {"V_D"  : "V(d1)",
          "V_G"  : "V(g1)",
          "I_DS" : "@M1[id]",
          "g_m2"  : "@M1[gm]",
          "c_iss2": "@M1[cgg]"}

fileName  = "kicad/Libs/biasN.kicad_sch"
sl.makeCircuit(fileName, language="SPICE")
op_info_2 = sl.ngspice2traces(sl.ini.cir_path + "biasN", simCmd, names, parList=params)
gm_ID = op_info_2["g_m2"]/op_info_2["I_DS"]
sl.eqn2html("g_m/I_D", gm_ID)

Creating netlist of kicad/Libs/biasN.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/Libs/biasN.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/biasN.svg'.
Done.


## Bandwidth and accuracy check of dual-stage amplifier

In [157]:
cir = sl.makeCircuit("kicad/A1/dualStageSimple.kicad_sch")
sl.img2html("dualStageSimple.svg", 800)

Creating netlist of kicad/A1/dualStageSimple.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/dualStageSimple.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/dualStageSimple.svg'.
Done.
Checking netlist: cir/dualStageSimple.cir


### Define the circuit parameters
To obtain equal noise performance as the CS stage, the small-signal parameters of the differential pair must equal those of the CS stage. This is at the costs of four times current and area.

In [158]:
sl.specs2circuit(specs, cir)
cir.defPars(op_info)
cir.defPars(op_info_2)
# Check for numeric values
sl.elementData2html(cir)

RefDes,Nodes,Refs,Model,Param,Symbolic,Numeric
C1,in fb,,C,value,$c_{iss}$,$16.94\cdot 10^{-15}$
,,,,vinit,$0$,$0$
C2,out 0,,C,value,$C_{L}$,$10\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C3,out fb,,C,value,$C_{i}$,$2.142\cdot 10^{-9}$
,,,,vinit,$0$,$0$
C4,in 0,,C,value,$C_{s}$,$9.382\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C5,int 0,,C,value,$c_{iss2}$,$3.924\cdot 10^{-15}$
,,,,vinit,$0$,$0$


## Loop gain analysis
### Pole-zero analysis
Pole-zero analysis of the loop gain yields the DC loop gain and its poles and zeros. From this data, we can analyse
the accuracy and the bandwidth of the amplifier.

In [159]:
PZresult = sl.doPZ(cir, pardefs="circuit", transfer="loopgain")
sl.pz2html(PZresult)

pole,Re [Hz],Im [Hz],Mag [Hz],Q
p1,$-597.2$,,$597.2$,
p2,$-106.5\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7077$
p3,$-106.5\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7077$
p4,$-2.128\cdot 10^{6}$,,$2.128\cdot 10^{6}$,
p5,$-40.12\cdot 10^{6}$,,$40.12\cdot 10^{6}$,
zero,Re [Hz],Im [Hz],Mag [Hz],Q
z1,$-600$,,$600$,
z2,$-106.6\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7071$
z3,$-106.6\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7071$


### Symbolic expression of DC loopgain

In [160]:
L_DC = sl.doLaplace(cir, transfer="loopgain").laplace.subs(sl.ini.laplace, 0)
sl.eqn2html("L_DC", L_DC)

### Plots asymptotic-gain model

In [161]:
AG = sl.doLaplace(cir, transfer="asymptotic", pardefs="circuit")
LG = sl.doLaplace(cir, transfer="loopgain",   pardefs="circuit")
SG = sl.doLaplace(cir, transfer="servo",      pardefs="circuit")
DG = sl.doLaplace(cir, transfer="direct",     pardefs="circuit")
GG = sl.doLaplace(cir, pardefs="circuit")
transfers = [AG, LG, SG, DG, GG]
sl.plotSweep("feedbackMag2", "Magnitude plots", transfers, 0.1, 1e7, 500, sweepScale="k")
sl.img2html("feedbackMag2.svg", 600)

### Check stability

In [162]:
PZ_GG = sl.doPZ(cir, pardefs="circuit", numeric=True)
sl.pz2html(PZ_GG)

pole,Re [Hz],Im [Hz],Mag [Hz],Q
p1,$-600.0$,,$600.0$,
p2,$-106.6\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7071$
p3,$-106.6\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7071$
p4,$-21.12\cdot 10^{6}$,$93.60\cdot 10^{6}$,$95.95\cdot 10^{6}$,$2.271$
p5,$-21.12\cdot 10^{6}$,$-93.60\cdot 10^{6}$,$95.95\cdot 10^{6}$,$2.271$
zero,Re [Hz],Im [Hz],Mag [Hz],Q
z1,$-10.51\cdot 10^{3}$,,$10.51\cdot 10^{3}$,
z2,$-20.06\cdot 10^{6}$,$2.324\cdot 10^{9}$,$2.324\cdot 10^{9}$,$57.95$
z3,$-20.06\cdot 10^{6}$,$-2.324\cdot 10^{9}$,$2.324\cdot 10^{9}$,$57.95$


In [163]:
# Check high-frequency plot
sl.plotSweep("gainMag-hf", "Magnitude plot gain (HF)", GG, 0.1, 200, 200, sweepScale="M")
sl.img2html("gainMag-hf.svg", 600)

## Dual stage with EKV small-signal model equations

In [164]:
EKVcir = sl.makeCircuit("kicad/A1/dualStageEKV.kicad_sch")
sl.img2html("dualStageEKV.svg", 600)

Creating netlist of kicad/A1/dualStageEKV.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/dualStageEKV.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/dualStageEKV.svg'.
Done.
Checking netlist: cir/dualStageEKV.cir


### Define the circuit parameters

In [165]:
ID1 = 2*op_info["I_DS"]
W1  = 2 * spec_dict["W_iP2"].value
L1  = 2 * spec_dict["L_iP2"].value

specs.append(sl.specItem(
    "ID1",
    value       = -ID1,
    description = "Input stage operating current",
    units       = "A",
    specType    = "design"))
specs.append(sl.specItem(
    "W1",
    value       = W1,
    description = "Input stage transistor width",
    units       = "m",
    specType    = "design"))
specs.append(sl.specItem(
    "L1",
    value       = L1,
    description = "Input stage transistor length",
    units       = "m",
    specType    = "design"))
specs.append(sl.specItem(
    "ID2",
    value       = I_D2,
    description = "Output stage operating current",
    units       = "A",
    specType    = "design"))
specs.append(sl.specItem(
    "W2",
    value       = W2,
    description = "Output stage transistor width",
    units       = "m",
    specType    = "design"))
specs.append(sl.specItem(
    "L2",
    value       = L2,
    description = "Output stage transistor length",
    units       = "m",
    specType    = "design"))
             
sl.specs2circuit(specs, EKVcir)
sl.elementData2html(EKVcir)

RefDes,Nodes,Refs,Model,Param,Symbolic,Numeric
C2,out 0,,C,value,$C_{L}$,$10\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C3,out 4,,C,value,$C_{i}$,$2.142\cdot 10^{-9}$
,,,,vinit,$0$,$0$
C4,in 0,,C,value,$C_{s}$,$9.382\cdot 10^{-12}$
,,,,vinit,$0$,$0$
Cdb_M1_X2,out 0,,C,value,$c_{db X2}$,$1.800\cdot 10^{-15}$
Cdd_M1_X1,3 0,,C,value,$0.5 c_{db X1}$,$2.097\cdot 10^{-15}$
Cdg1_M1_X1,3 in,,C,value,$c_{dg X1}$,$6.991\cdot 10^{-15}$
Cdg2_M1_X1,0 4,,C,value,$c_{dg X1}$,$6.991\cdot 10^{-15}$


In [166]:
EKV_PZ_LG = sl.doPZ(EKVcir, pardefs="circuit", transfer="loopgain", numeric=True)
# Save this information for later use
%store EKV_PZ_LG
sl.pz2html(EKV_PZ_LG)

Stored 'EKV_PZ_LG' (instruction)


pole,Re [Hz],Im [Hz],Mag [Hz],Q
p1,$-745.4$,,$745.4$,
p2,$-92.16\cdot 10^{3}$,$105.1\cdot 10^{3}$,$139.8\cdot 10^{3}$,$0.7585$
p3,$-92.16\cdot 10^{3}$,$-105.1\cdot 10^{3}$,$139.8\cdot 10^{3}$,$0.7585$
p4,$-2.570\cdot 10^{6}$,$414.5\cdot 10^{3}$,$2.603\cdot 10^{6}$,$0.5065$
p5,$-2.570\cdot 10^{6}$,$-414.5\cdot 10^{3}$,$2.603\cdot 10^{6}$,$0.5065$
zero,Re [Hz],Im [Hz],Mag [Hz],Q
z1,$-600$,,$600$,
z2,$-106.6\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7074$
z3,$-106.6\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.8\cdot 10^{3}$,$0.7074$
z4,$-3.670\cdot 10^{9}$,,$3.670\cdot 10^{9}$,


### Mid-band loop gain and servo bandwidth

In [167]:
EKV_LG = sl.doLaplace(EKVcir, pardefs="circuit", transfer="loopgain", numeric=True)
servo_info = sl.findServoBandwidth(EKV_LG.laplace)
sl.eqn2html("L_MB", -servo_info["mbv"])

The loop gain has five dominant poles and three dominant zeros. Hence, the order of the servo function at low-pass cut-off
is -2:

In [168]:
servo_info["lpo"]

-2.0

Since the complex poles and the complex zeros of the loopgain almost coincide, the bandwidth $B_f$ can be estimated from:
\begin{equation}
B_f = \sqrt{L_{DC}\frac{p_1 p_4 p_5}{z_1}}
\end{equation}

In [169]:
L_poles = EKV_PZ_LG.poles
L_zeros = EKV_PZ_LG.zeros
L_DC    = EKV_PZ_LG.DCvalue
Bf      = N(sqrt((1-L_DC)*L_poles[0]*L_poles[3]*L_poles[4]/L_zeros[0])/2/pi, 4)
Bf

4.141e+7

This closely natches the value found by **findServoBandwidth()**:

In [170]:
sl.eqn2html("B_f", servo_info["lpf"], units= "Hz")

### Plots asymptotic-gain model

In [171]:
EKV_AG = sl.doLaplace(EKVcir, transfer="asymptotic", pardefs="circuit", numeric=True)
EKV_SG = sl.doLaplace(EKVcir, transfer="servo",      pardefs="circuit", numeric=True)
EKV_DG = sl.doLaplace(EKVcir, transfer="direct",     pardefs="circuit", numeric=True)
EKV_GG = sl.doLaplace(EKVcir, pardefs="circuit", numeric=True)
EKV_transfers = [EKV_AG, EKV_LG, EKV_SG, EKV_DG, EKV_GG]
sl.plotSweep("EKV_feedbackMag3", "Magnitude plots", EKV_transfers, 0.1, 10e6, 500, sweepScale="k")
sl.img2html("EKV_feedbackMag3.svg", 600)

### Check frequency stability

In [172]:
EKV_PZ_GG = sl.doPZ(EKVcir, pardefs="circuit", numeric=True)
sl.pz2html(EKV_PZ_GG)

pole,Re [Hz],Im [Hz],Mag [Hz],Q
p1,$-600.6$,,$600.6$,
p2,$-106.5\cdot 10^{3}$,$106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7076$
p3,$-106.5\cdot 10^{3}$,$-106.6\cdot 10^{3}$,$150.7\cdot 10^{3}$,$0.7076$
p4,$-2.756\cdot 10^{6}$,$38.29\cdot 10^{6}$,$38.39\cdot 10^{6}$,$6.966$
p5,$-2.756\cdot 10^{6}$,$-38.29\cdot 10^{6}$,$38.39\cdot 10^{6}$,$6.966$
zero,Re [Hz],Im [Hz],Mag [Hz],Q
z1,$118.8\cdot 10^{6}$,$559.2\cdot 10^{6}$,$571.7\cdot 10^{6}$,$2.406$
z2,$118.8\cdot 10^{6}$,$-559.2\cdot 10^{6}$,$571.7\cdot 10^{6}$,$2.406$
z3,$-10.51\cdot 10^{3}$,,$10.51\cdot 10^{3}$,


In [173]:
# Gain plot (LF) 
sl.plotSweep("EKV_gainMag-lf", "Magnitude plot gain (LF)", EKV_GG, 100, 10e3, 200)
sl.img2html("EKV_gainMag-lf.svg", 600)

In [174]:
# Check high-frequency plot
sl.plotSweep("EKV_gainMag-hf", "Magnitude plot gain (HF)", EKV_GG, 0.1, 200, 200, sweepScale="M")
sl.img2html("EKV_gainMag-hf.svg", 600)

In [175]:
# Check very-high-frequency plot
sl.plotSweep("EKV_gainMag-vhf", "Magnitude plot gain (VHF)", EKV_GG, 100, 1e3, 200, sweepScale="M")
sl.img2html("EKV_gainMag-vhf.svg", 600)

In [176]:
sl.specs2csv(specs, "dualStage.csv")

### Questions

1. Discuss possible differences between the results obtained with the simple model and those obtained with the EKV model.
2. Are the circuits stable?
3. What is the cause of the peaking around 40MHz?
5. Do you suggest any frequency compensation?
6. If so, suggest a method and implement it.
7. What causes the high-frequency positive zeros?
8. Do we need to compensate for them?